In [0]:
# Databricks notebook: 03_data_quality
# Qualidade de Dados - player_season_club (2014)

from pyspark.sql import functions as F

silver = spark.table("estudo_estrangeiros.silver.player_season_club")
total_rows = silver.count()
print(f"Total de linhas: {total_rows}")

# --- Completude: % de nulos por coluna ---
print("\n=== Completude (% nulos por coluna) ===")
null_pct = silver.select([
    (F.sum(F.col(c).isNull().cast("int")) / total_rows * 100).alias(c)
    for c in silver.columns
])
display(null_pct)

# --- Unicidade: grain player_id + season + squad deve ser único ---
dup_count = (
    silver.groupBy("player_id", "season", "squad")
    .count()
    .filter("count > 1")
    .count()
)
print(f"\n=== Unicidade ===\nCombinações duplicadas de (player_id, season, squad): {dup_count}")

# --- Consistência/Acurácia: faixas plausíveis ---
print("\n=== Consistência / Acurácia ===")
inconsistencias = silver.filter(
    (F.col("age") < 14) | (F.col("age") > 46) |          # idade fora de faixa plausível
    (F.col("minutes") < 0) | (F.col("minutes") > 3420) | # > 38 jogos x 90min
    (F.col("starts") > F.col("matches_played")) |        # titularidades > jogos
    (F.col("goals") < 0) | (F.col("assists") < 0)
)
print(f"Linhas com valores fora do esperado: {inconsistencias.count()}")
display(inconsistencias)

# --- Outliers: top 10 artilheiros/minutagem (checagem visual de sanidade) ---
print("\n=== Outliers (sanity check visual) ===")
display(silver.orderBy(F.col("goals").desc()).limit(10))
display(silver.orderBy(F.col("minutes").desc()).limit(10))


Total de linhas: 8887

=== Completude (% nulos por coluna) ===


player_id,player_name,nation_raw,position_raw,squad,age_fbref,birth_year,matches_played,starts,minutes,goals,assists,season,nation_code,is_brazilian,position_primary,position_secondary,age,age_band
0.0,0.0,0.03375717339934736,0.022504782266231572,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.03375717339934736,0.03375717339934736,0.022504782266231572,79.28434792393384,0.0,0.0



=== Unicidade ===
Combinações duplicadas de (player_id, season, squad): 0

=== Consistência / Acurácia ===
Linhas com valores fora do esperado: 0


player_id,player_name,nation_raw,position_raw,squad,age_fbref,birth_year,matches_played,starts,minutes,goals,assists,season,nation_code,is_brazilian,position_primary,position_secondary,age,age_band



=== Outliers (sanity check visual) ===


player_id,player_name,nation_raw,position_raw,squad,age_fbref,birth_year,matches_played,starts,minutes,goals,assists,season,nation_code,is_brazilian,position_primary,position_secondary,age,age_band
9ccf4e14,Germán Cano,ar ARG,FW,Fluminense,33,1988,38,37,3081,26,3,2022,ARG,false,FW,null,34,30+
72d0e1b6,Gabriel Barbosa,br BRA,FWMF,Flamengo,22,1996,29,29,2537,24,9,2019,BRA,true,FW,MF,23,<=23
5edff24f,Bruno Henrique,br BRA,MFFW,Flamengo,28,1990,33,29,2570,22,3,2019,BRA,true,MF,FW,29,24-29
0cf465f2,Kaio Jorge,br BRA,FW,Cruzeiro,23,2002,33,33,2588,21,8,2025,BRA,true,FW,null,23,<=23
53070b63,Paulinho,br BRA,FWMF,Atlético Mineiro,22,2000,36,35,2991,20,2,2023,BRA,true,FW,MF,23,<=23
420c1246,Ricardo Oliveira,br BRA,FW,Santos,34,1980,32,31,2737,20,2,2015,BRA,true,FW,null,35,30+
33a1f72e,Hulk,br BRA,FW,Atlético Mineiro,34,1986,35,33,2921,19,7,2021,BRA,true,FW,null,35,30+
ed074559,Pedro Raul,br BRA,FW,Goiás,25,1996,34,33,2745,19,2,2022,BRA,true,FW,null,26,24-29
24ce161c,Luciano,br BRA,FWMF,São Paulo,26,1993,31,29,2492,18,3,2020,BRA,true,FW,MF,27,24-29
be8d02dd,Giorgian de Arrascaeta,uy URU,MF,Flamengo,30,1994,33,30,2350,18,14,2025,URU,false,MF,null,31,30+


player_id,player_name,nation_raw,position_raw,squad,age_fbref,birth_year,matches_played,starts,minutes,goals,assists,season,nation_code,is_brazilian,position_primary,position_secondary,age,age_band
15825726,Fábio,br BRA,GK,Fluminense,41,1980,38,38,3420,0,0,2022,BRA,true,GK,null,42,30+
fba79ac8,Tiago Volpi,br BRA,GK,São Paulo,30,1990,38,38,3420,0,0,2021,BRA,true,GK,null,31,30+
15825726,Fábio,br BRA,GK,Fluminense,44,1980,38,38,3420,0,0,2025,BRA,true,GK,null,45,30+
cfc1d935,Victor Leandro Bagy,br BRA,GK,Atlético Mineiro,32,1983,38,38,3420,0,0,2015,BRA,true,GK,null,32,30+
4729d71a,João Ricardo Riedi,br BRA,GK,Fortaleza,35,1988,38,38,3420,0,0,2024,BRA,true,GK,null,36,30+
fba79ac8,Tiago Volpi,br BRA,GK,São Paulo,29,1990,38,38,3420,0,1,2020,BRA,true,GK,null,30,30+
f5dfc256,Léo Jardim,br BRA,GK,Vasco da Gama,27,1995,38,38,3420,0,0,2023,BRA,true,GK,null,28,24-29
f5dfc256,Léo Jardim,br BRA,GK,Vasco da Gama,28,1995,38,38,3420,0,0,2024,BRA,true,GK,null,29,24-29
8445554f,Tadeu,br BRA,GK,Goiás,29,1992,38,38,3420,0,0,2022,BRA,true,GK,null,30,30+
ab55df84,Vanderlei,br BRA,GK,Coritiba,29,1984,38,38,3420,0,0,2014,BRA,true,GK,null,30,30+


In [0]:
display(silver.filter(F.col("player_id").isNull() | F.col("player_name").isNull()))


player_id,player_name,nation_raw,position_raw,squad,age_fbref,birth_year,matches_played,starts,minutes,goals,assists,season,nation_code,is_brazilian,position_primary,position_secondary,age,age_band
null,null,null,null,null,null,null,null,null,null,null,null,2014,null,null,null,null,null,30+


In [0]:
from pyspark.sql import functions as F

# nossos números: minutos e jogadores por nacionalidade, calculados a partir do nível de jogador
calculado = (
    spark.table("estudo_estrangeiros.silver.player_season_club")
    .filter("nation_raw IS NOT NULL")
    .withColumn("cod_pais", F.lower(F.element_at(F.split("nation_raw", " "), 1)))
    .groupBy("season", "cod_pais")
    .agg(
        F.sum("minutes").alias("minutos_calculados"),
        F.countDistinct("player_id").alias("jogadores_calculados"),
    )
)

# números que o próprio FBref publica já agregados, na tabela nationalities
publicado = (
    spark.table("estudo_estrangeiros.bronze.nationalities")
    .withColumn("cod_pais", F.lower(F.element_at(F.split(F.col("Nation"), " "), 1)))
    .select(
        F.expr("try_cast(_season AS INT)").alias("season"),
        "cod_pais",
        F.expr("try_cast(try_cast(`Min` AS DOUBLE) AS INT)").alias("minutos_publicados"),
        F.expr("try_cast(try_cast(`#_Players` AS DOUBLE) AS INT)").alias("jogadores_publicados"),
    )
)

validacao = (
    calculado.join(publicado, ["season", "cod_pais"], "full_outer")
    .withColumn("dif_minutos", F.col("minutos_calculados") - F.col("minutos_publicados"))
    .withColumn("dif_pct", F.round(F.abs(F.col("dif_minutos")) / F.col("minutos_publicados") * 100, 2))
)

print("Combinações temporada × nacionalidade comparadas:", validacao.count())
print("Divergência de minutos acima de 1%:", validacao.filter("dif_pct > 1").count())
print("Sem correspondência de um dos lados:", validacao.filter("minutos_calculados IS NULL OR minutos_publicados IS NULL").count())

display(validacao.orderBy(F.col("dif_pct").desc_nulls_first()).limit(20))


Combinações temporada × nacionalidade comparadas: 250
Divergência de minutos acima de 1%: 0
Sem correspondência de um dos lados: 12


season,cod_pais,minutos_calculados,jogadores_calculados,minutos_publicados,jogadores_publicados,dif_minutos,dif_pct
2016,pt,null,null,null,1,null,null
2025,ng,null,null,null,1,null,null
2026,ci,null,null,null,1,null,null
2018,jp,null,null,null,1,null,null
2020,at,null,null,null,1,null,null
2025,gh,null,null,null,2,null,null
2018,my,null,null,871,1,null,null
2014,au,null,null,null,1,null,null
2023,ni,null,null,null,1,null,null
2015,jp,null,null,null,1,null,null


In [0]:
display(
    spark.table("estudo_estrangeiros.silver.player_season_club")
    .filter("nation_raw IS NULL")
    .groupBy("season")
    .count()
    .orderBy("season")
)


season,count
2016,1
2018,1
2020,1


In [0]:
display(
    spark.table("estudo_estrangeiros.silver.player_season_club")
    .filter("position_raw IS NULL")
    .groupBy("season")
    .count()
    .orderBy("season")
)

display(
    spark.table("estudo_estrangeiros.silver.player_season_club")
    .filter("position_raw IS NULL")
    .select("season", "player_name", "squad", "minutes", "position_raw")
    .orderBy("season")
)



season,count
2020,2


season,player_name,squad,minutes,position_raw
2020,Flávio,Bahia,10,null
2020,Danilo,Atl. Goianiense,20,null


In [0]:
display(
    spark.table("estudo_estrangeiros.silver.player_season_club")
    .filter("nation_raw IS NULL")
    .groupBy("season")
    .count()
    .orderBy("season")
)

display(
    spark.table("estudo_estrangeiros.silver.player_season_club")
    .filter("nation_raw IS NULL")
    .select("season", "player_name", "squad", "minutes", "nation_raw")
    .orderBy("season")
)



season,count
2016,1
2018,1
2020,1


season,player_name,squad,minutes,nation_raw
2016,Artur,Palmeiras,4,null
2018,Artur,Palmeiras,115,null
2020,Danilo,Atl. Goianiense,20,null
